In [49]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [50]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [51]:
unique_isin = tuple(isin['ISIN'])

In [52]:
isin['ISIN'].str[:2].unique()

array(['DE', 'IT', 'FR', 'ES'], dtype=object)

In [53]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [54]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [55]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [56]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

# time series relationship

In [57]:
# Data prep
query = f"""

SELECT 
    s.lender_id AS fund_id,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND s_borrower.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.lender_id
ORDER BY lending_volume DESC
"""

df_funds = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_21164\3674722318.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_funds = pd.read_sql_query(query, cnxn)


In [58]:
df_funds.head(20)

,fund_id,lending_volume
0,FJV0KIIMRXLMWV5QT846,1.680436e+13
1,P5XEQYFJP74DYQX88M80,1.143076e+13
2,2CNR4I7RPCUNYMQ52H12,1.038093e+13
3,5493008P9DJX0WNGA303,8.923998e+12
4,549300HY72WJJ6KYOB71,6.703034e+12
5,549300RXYC2IDG39CW23,6.053766e+12
6,DS1LD4KRZVNKQC3WP076,5.315622e+12
7,549300KM4DRKLEPZUO76,4.827628e+12
8,549300APE0JGFJ9JMB36,4.734197e+12
9,549300OR6B15UIZ9FZ95,4.475655e+12


In [59]:
large_funds = tuple(['FJV0KIIMRXLMWV5QT846', 'P5XEQYFJP74DYQX88M80', '2CNR4I7RPCUNYMQ52H12'])
mid_funds = tuple(['549300NFUQNQSETLCP08', '22KMWLRMY0MUEN0W5D90'])
funds = large_funds + mid_funds

In [60]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND s_lender.sector = 'DEALER'
    AND s.borrower_id IN {funds}
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id
ORDER BY s.business_date, s.borrower_id, s.lender_id
"""

df_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_21164\3973239351.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing = pd.read_sql_query(query, cnxn)


In [61]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND s_borrower.sector = 'DEALER'
    AND s.lender_id IN {funds}
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id
ORDER BY s.business_date, s.lender_id, s.borrower_id
"""

df_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_21164\54963593.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending = pd.read_sql_query(query, cnxn)


In [62]:
df_net = df_borrowing.merge(df_lending, on= ['business_date', 'fund_id', 'dealer_id'], how = 'outer')
df_net[['borrowing_volume', 'lending_volume']] = df_net[['borrowing_volume', 'lending_volume']].fillna(0)
df_net['net_position'] = (df_net['borrowing_volume'] - df_net['lending_volume'])/1e9
df_net['business_date'] = pd.to_datetime(df_net['business_date'])

In [63]:
# Data prep
query = f"""

SELECT 
    s.lender_id AS dealer_id,
    s.lender_name AS dealer_name,
    COUNT(*)                                                                            AS n_obs
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'DEALER'
    AND s.lender_name IS NOT NULL
    AND s.security_isin IN {unique_isin}
GROUP BY s.lender_id, s.lender_name
"""

df_names_lender = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_21164\2124540020.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_names_lender = pd.read_sql_query(query, cnxn)


In [64]:
# Data prep
query = f"""

SELECT 
    s.borrower_id AS dealer_id,
    s.borrower_name AS dealer_name,
    COUNT(*)                                                                            AS n_obs
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'DEALER'
    AND s.borrower_name IS NOT NULL
    AND s.security_isin IN {unique_isin}
GROUP BY s.borrower_id, s.borrower_name
"""

df_names_borrower = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_21164\893495975.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_names_borrower = pd.read_sql_query(query, cnxn)


In [65]:
df_names = pd.concat([df_names_lender, df_names_borrower])
df_names = df_names.groupby(['dealer_id', 'dealer_name'])['n_obs'].sum().reset_index()
dealer_names = df_names.sort_values('n_obs').drop_duplicates('dealer_id', keep='last').set_index('dealer_id')['dealer_name']

In [66]:
fund_labels = {large_funds[0]: 'Large fund 1', large_funds[1]: 'Large fund 2', large_funds[2]: 'Large fund 3',
               mid_funds[0]: 'Mid-sized fund 1', mid_funds[1]: 'Mid-sized fund 2'}

In [ ]:
for fund in funds:
    pivot = df_net[df_net['fund_id'] == fund].pivot_table(index='business_date', columns='dealer_id', values='net_position')
    pivot = pivot.rename(columns=dealer_names)

    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(ax=ax)
    ax.plot(pivot.index, pivot.sum(axis=1), color='black', linewidth=2, label='Fund total')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Business Date')
    ax.set_ylabel('Net position (borrowing - lending) in bln')
    ax.set_title(fund_labels[fund])
    ax.legend(loc='best', ncol=2)
    plt.tight_layout()
    plt.show()

# dealer concentration

In [ ]:
# Data prep
query = f"""

SELECT 
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'DEALER'
    AND s_borrower.sector = 'HF'
    AND s.security_isin IN {unique_isin}
GROUP BY s.lender_id
"""

df_dealer_lending = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'DEALER'
    AND s_lender.sector = 'HF'
    AND s.security_isin IN {unique_isin}
GROUP BY s.borrower_id
"""

df_dealer_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
df_dealer_vol = pd.concat([df_dealer_lending, df_dealer_borrowing])
df_dealer_vol = df_dealer_vol.groupby('dealer_id')['volume'].sum().sort_values(ascending=False)
dealer_shares = (df_dealer_vol / df_dealer_vol.sum()).rename(index=dealer_names)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(dealer_shares.index, dealer_shares.values, color='#2166ac')
ax.set_ylabel('Share of total volume')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()